In [1]:
from pathlib import Path
import pynq
import time

In [2]:
def load_csv_ints(file_path: Path) -> list[int]:
    values: list[int] = []
    with file_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            values.extend(int(token.strip()) for token in line.split(",") if token.strip())
    return values


def check(labels, recv_buff, n=1):
    correct = True
    n_labels = len(labels)
    for i in range(n):
        for idx, (actual, predicted) in enumerate(zip(labels, recv_buff[i * n_labels:])):
            if (actual != predicted):
                print(f"Mismatch at idx {i * n_labels + idx}: actual={actual}, predicted={predicted}")
                correct = False
    return correct
                

In [3]:
data_dir = Path("./data")
ordered_files = ["w_hid.csv", "w_out.csv"]
payload: list[int] = []

X = load_csv_ints(data_dir / "X.csv")
labels = load_csv_ints(Path("./data/labels.csv"))

for name in ordered_files:
    payload.extend(load_csv_ints(data_dir / name))

payload.extend(X)


In [4]:
ov = pynq.Overlay("project_hw.xsa")


In [5]:
names = ["hls", "hdl"]
dmas = [ov.dma_hls_ip, ov.dma_hdl_ip]
send_chans = [dma.sendchannel for dma in dmas]
recv_chans = [dma.recvchannel for dma in dmas]

In [8]:
send_buff = pynq.allocate(shape=(2048,))
recv_buff = pynq.allocate(shape=(2048,))

In [11]:
for index, value in enumerate(payload):
    send_buff[index] = value

print("=== 64 x 7 X DMA Transaction Comparison ===")
for (name, send, recv) in zip(names, send_chans, recv_chans):
    print(f"=== {name} ===")
    start = time.time()
    send.transfer(send_buff[: len(payload)])
    recv.transfer(recv_buff)
    send.wait()
    recv.wait()
    end = time.time()
    print("--- Checking Results ---")
    correct = check(labels, recv_buff)
    print("--- Summary ---")
    print(f"Transaction duation: {end - start}")
    print(f"Correctness check: {'PASS' if correct else  'FAIL'}")
    print("=== END ===\n")
    

=== 64 x 7 X DMA Transaction Comparison ===
=== hls ===


RuntimeError: DMA channel not started